# 03b — Audit del target: Tumor + Healthy

## Obiettivo

Verificare se il task di classificazione utilizzato finora coincide con
l'obiettivo del progetto: associare un sample eccDNA al tipo di tumore,
includendo anche la classe Healthy.

Il dataset completo contiene 18 classi, alcune delle quali sembrano
rappresentare patologie non oncologiche.

Prima di modificare il dataset viene quindi analizzata la variabile
`disease_group` per verificare come le classi siano categorizzate
nel metadata originale.

Il dataset originale non viene modificato.
Eventualmente verrà creato un nuovo manifest contenente solamente:

- classi appartenenti al gruppo cancer;
- classe Healthy.

Lo split basato su `split_cluster` viene mantenuto invariato.

In [1]:
# ============================================================
# CELL 2 — IMPORT E DATA
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


MANIFEST_PATH = (
    PROCESSED_DIR
    / "siamese_primary_manifest.tsv"
)


metadata = pd.read_csv(
    MANIFEST_PATH,
    sep="\t",
    dtype={
        "id": str
    }
)


metadata["class_id"] = (
    metadata["class_id"]
    .astype(int)
)


print(
    "Sample totali:",
    len(metadata)
)

print(
    "Classi:",
    metadata["class_id"].nunique()
)

print(
    "Disease group:",
    metadata["disease_group"].unique()
)

Sample totali: 665681
Classi: 18
Disease group: <StringArray>
['cancer', 'healthy', 'non_cancer_disease']
Length: 3, dtype: str


In [2]:
# ============================================================
# CELL 3 — AUDIT DISEASE GROUP
# ============================================================

disease_audit_df = (

    metadata[
        [
            "class_id",
            "disease",
            "disease_clean",
            "disease_group"
        ]
    ]

    .drop_duplicates()

    .sort_values(
        [
            "class_id",
            "disease_clean"
        ]
    )

    .reset_index(
        drop=True
    )
)


display(
    disease_audit_df
)

,class_id,disease,disease_clean,disease_group
0,0,Gastric cancer,gastric cancer,cancer
1,0,Stomach,gastric cancer,cancer
2,0,Stomach Cancer,gastric cancer,cancer
3,1,Health,healthy,healthy
4,1,Healthy,healthy,healthy
5,2,Ovarian Cancer,ovarian cancer,cancer
6,2,Ovarian cancer,ovarian cancer,cancer
7,3,Prostate Cancer,prostate cancer,cancer
8,3,Prostate cancer,prostate cancer,cancer
9,4,Colorectal cancer,colorectal cancer,cancer


In [3]:
# ============================================================
# CELL 4 — CONSISTENZA CLASS_ID / DISEASE_GROUP
# ============================================================

group_consistency = (

    metadata
    .groupby("class_id")
    .agg(
        disease_clean=(
            "disease_clean",
            lambda x: sorted(
                x.dropna().unique()
            )
        ),
        disease_group=(
            "disease_group",
            lambda x: sorted(
                x.dropna().unique()
            )
        ),
        n_samples=(
            "id",
            "size"
        )
    )
    .reset_index()
)


display(
    group_consistency
)


problematic_groups = (
    group_consistency[
        group_consistency[
            "disease_group"
        ].apply(len) != 1
    ]
)


print(
    "Classi con disease_group ambiguo:",
    len(problematic_groups)
)

,class_id,disease_clean,disease_group,n_samples
0,0,[gastric cancer],[cancer],391239
1,1,[healthy],[healthy],73995
2,2,[ovarian cancer],[cancer],35122
3,3,[prostate cancer],[cancer],34487
4,4,[colorectal cancer],[cancer],22933
5,5,[lymphoma],[cancer],20185
6,6,[hiv infectious disease],[non_cancer_disease],16315
7,7,[cervical adenocarcinoma],[cancer],15677
8,8,[leukemia],[cancer],13456
9,9,[primary pulmonary hypertension],[non_cancer_disease],8119


Classi con disease_group ambiguo: 0


In [4]:
# ============================================================
# CELL 5 — DISTRIBUZIONE DEI GRUPPI
# ============================================================

group_summary = (

    metadata
    .groupby(
        "disease_group"
    )
    .agg(
        n_samples=(
            "id",
            "size"
        ),
        n_classes=(
            "class_id",
            "nunique"
        )
    )
    .sort_values(
        "n_samples",
        ascending=False
    )
    .reset_index()
)


display(
    group_summary
)

,disease_group,n_samples,n_classes
0,cancer,551942,11
1,healthy,73995,1
2,non_cancer_disease,39744,6


In [5]:
# ============================================================
# CELL 6 — TUMOR + HEALTHY TASK
# ============================================================

TARGET_GROUPS = [
    "cancer",
    "healthy"
]


tumor_healthy_mask = (
    metadata["disease_group"]
    .isin(TARGET_GROUPS)
)


tumor_healthy_metadata = (
    metadata[
        tumor_healthy_mask
    ]
    .copy()
    .reset_index(drop=True)
)


excluded_metadata = (
    metadata[
        ~tumor_healthy_mask
    ]
    .copy()
)


print("=" * 70)
print("TUMOR + HEALTHY TASK")
print("=" * 70)

print(
    "Sample inclusi:",
    len(tumor_healthy_metadata)
)

print(
    "Classi incluse:",
    tumor_healthy_metadata[
        "class_id"
    ].nunique()
)

print()

print(
    "Sample esclusi:",
    len(excluded_metadata)
)

print(
    "Classi escluse:",
    excluded_metadata[
        "class_id"
    ].nunique()
)

TUMOR + HEALTHY TASK
Sample inclusi: 625937
Classi incluse: 12

Sample esclusi: 39744
Classi escluse: 6


In [6]:
# ============================================================
# CELL 7 — INCLUDED / EXCLUDED CLASSES
# ============================================================

included_classes_df = (

    tumor_healthy_metadata[
        [
            "class_id",
            "disease_clean",
            "disease_group"
        ]
    ]

    .drop_duplicates()

    .sort_values(
        "class_id"
    )

    .reset_index(
        drop=True
    )
)


excluded_classes_df = (

    excluded_metadata[
        [
            "class_id",
            "disease_clean",
            "disease_group"
        ]
    ]

    .drop_duplicates()

    .sort_values(
        "class_id"
    )

    .reset_index(
        drop=True
    )
)


print("CLASSI INCLUSE")
display(
    included_classes_df
)


print()
print("CLASSI ESCLUSE")
display(
    excluded_classes_df
)

CLASSI INCLUSE


,class_id,disease_clean,disease_group
0,0,gastric cancer,cancer
1,1,healthy,healthy
2,2,ovarian cancer,cancer
3,3,prostate cancer,cancer
4,4,colorectal cancer,cancer
5,5,lymphoma,cancer
6,7,cervical adenocarcinoma,cancer
7,8,leukemia,cancer
8,11,hypopharyngeal squamous cell carcinoma,cancer
9,12,glioblastoma cancer,cancer



CLASSI ESCLUSE


,class_id,disease_clean,disease_group
0,6,hiv infectious disease,non_cancer_disease
1,9,primary pulmonary hypertension,non_cancer_disease
2,10,cataract,non_cancer_disease
3,14,dilated cardiomyopathy,non_cancer_disease
4,16,chronic kidney disease,non_cancer_disease
5,17,branchio-oculo-facial syndrome (bofs),non_cancer_disease


In [7]:
# ============================================================
# CELL 8 — NEW CLASS MAPPING
# ============================================================

class_mapping = (

    included_classes_df[
        [
            "class_id",
            "disease_clean",
            "disease_group"
        ]
    ]

    .rename(
        columns={
            "class_id":
                "original_class_id"
        }
    )

    .sort_values(
        "original_class_id"
    )

    .reset_index(
        drop=True
    )
)


class_mapping[
    "class_id"
] = np.arange(
    len(class_mapping),
    dtype=np.int64
)


old_to_new_class_id = dict(
    zip(
        class_mapping[
            "original_class_id"
        ],
        class_mapping[
            "class_id"
        ]
    )
)


display(
    class_mapping[
        [
            "class_id",
            "original_class_id",
            "disease_clean",
            "disease_group"
        ]
    ]
)

,class_id,original_class_id,disease_clean,disease_group
0,0,0,gastric cancer,cancer
1,1,1,healthy,healthy
2,2,2,ovarian cancer,cancer
3,3,3,prostate cancer,cancer
4,4,4,colorectal cancer,cancer
5,5,5,lymphoma,cancer
6,6,7,cervical adenocarcinoma,cancer
7,7,8,leukemia,cancer
8,8,11,hypopharyngeal squamous cell carcinoma,cancer
9,9,12,glioblastoma cancer,cancer


In [8]:
# ============================================================
# CELL 9 — BUILD TUMOR + HEALTHY MANIFEST
# ============================================================

tumor_healthy_manifest = (
    tumor_healthy_metadata
    .copy()
)


# Manteniamo traccia della label originale
tumor_healthy_manifest[
    "original_class_id"
] = (
    tumor_healthy_manifest[
        "class_id"
    ]
)


# Nuova label 0...11
tumor_healthy_manifest[
    "class_id"
] = (

    tumor_healthy_manifest[
        "original_class_id"
    ]

    .map(
        old_to_new_class_id
    )

    .astype(
        np.int64
    )
)


print(
    "Classi finali:",
    tumor_healthy_manifest[
        "class_id"
    ].nunique()
)


display(
    tumor_healthy_manifest[
        [
            "id",
            "disease_clean",
            "disease_group",
            "original_class_id",
            "class_id",
            "split_cluster"
        ]
    ]
    .head()
)

Classi finali: 12


,id,disease_clean,disease_group,original_class_id,class_id,split_cluster
0,CircleBaseV2_000726522,gastric cancer,cancer,0,0,train
1,CircleBaseV2_000119689,gastric cancer,cancer,0,0,train
2,CircleBaseV2_002112184,gastric cancer,cancer,0,0,train
3,CircleBaseV2_001452203,gastric cancer,cancer,0,0,train
4,CircleBaseV2_000827693,gastric cancer,cancer,0,0,train


In [9]:
# ============================================================
# CELL 10 — SPLIT AUDIT
# ============================================================

split_summary = (

    tumor_healthy_manifest

    .groupby(
        [
            "class_id",
            "disease_clean",
            "split_cluster"
        ]
    )

    .size()

    .rename(
        "n_samples"
    )

    .reset_index()
)


split_pivot = (

    split_summary

    .pivot_table(
        index=[
            "class_id",
            "disease_clean"
        ],
        columns=
            "split_cluster",
        values=
            "n_samples",
        fill_value=0
    )

    .reset_index()
)


display(
    split_pivot
)

split_cluster,class_id,disease_clean,test,train,val
0,0,gastric cancer,128668.0,10000.0,252571.0
1,1,healthy,20852.0,10000.0,43143.0
2,2,ovarian cancer,8355.0,10000.0,16767.0
3,3,prostate cancer,8263.0,10000.0,16224.0
4,4,colorectal cancer,4422.0,10000.0,8511.0
5,5,lymphoma,3409.0,10000.0,6776.0
6,6,cervical adenocarcinoma,1958.0,10000.0,3719.0
7,7,leukemia,1138.0,10000.0,2318.0
8,8,hypopharyngeal squamous cell carcinoma,288.0,5052.0,585.0
9,9,glioblastoma cancer,296.0,4880.0,505.0


In [10]:
# ============================================================
# CELL 11 — SAVE TUMOR + HEALTHY MANIFEST
# ============================================================

TUMOR_HEALTHY_MANIFEST_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_manifest.tsv"
)


TUMOR_HEALTHY_CLASS_MAPPING_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_class_mapping.tsv"
)


tumor_healthy_manifest.to_csv(
    TUMOR_HEALTHY_MANIFEST_PATH,
    sep="\t",
    index=False
)


class_mapping.to_csv(
    TUMOR_HEALTHY_CLASS_MAPPING_PATH,
    sep="\t",
    index=False
)


print(
    "Manifest:",
    TUMOR_HEALTHY_MANIFEST_PATH
)

print(
    "Mapping:",
    TUMOR_HEALTHY_CLASS_MAPPING_PATH
)

Manifest: D:\Daria\Desktop\eccdna_fcgr_siamese\data\processed\siamese_tumor_healthy_manifest.tsv
Mapping: D:\Daria\Desktop\eccdna_fcgr_siamese\data\processed\siamese_tumor_healthy_class_mapping.tsv
